# Gemma 4 Developer Agent — Post-Run Trace Analytics

Run this notebook once to generate two complementary views from the same normalized traces: a single-attempt explorer and a whole-batch report. Analysis is offline and read-only; no code from traces is executed. Open `../reports/itr-1/report.html` for one trace or `../reports/itr-1/batch_report.html` for the batch.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'agent_eval').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'agent_eval').is_dir():
    raise RuntimeError('Run this notebook from gemma4_eval or its notebooks folder')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

CONFIG = {
    'trace_dir': str(Path(__import__('os').environ.get('GEMMA_TRACE_DIR', next((path for path in [PROJECT_ROOT.parent / 'logs/remote/akshay/itr-1', Path('/Users/akshay/kaggle/wt/main/logs/remote/akshay/itr-1')] if path.is_dir()), PROJECT_ROOT.parent / 'logs/remote/akshay/itr-1')))),
    'trace_globs': ['**/model_trace.json', '**/trace_*.json'],
    'gold_dir': None,
    'repo_roots': {},  # repository name -> pristine baseline checkout
    'compare_dir': None,
    'output_dir': str(PROJECT_ROOT / 'reports' / 'itr-1'),
    'experiment_id': 'itr-1',
    'strict_schema': False,
    'redact_text': True,
    'include_reasoning_in_viewer': False,
}

## Load and normalize

Malformed inputs are reported per file. Unknown outcomes and unmatched results remain unknown. Duplicates remain separate attempts.

In [2]:
from agent_eval import load_experiment, compute_metrics, classify_failures, export_report
from agent_eval.plots import render_report

experiment = load_experiment(
    CONFIG['trace_dir'], experiment_id=CONFIG['experiment_id'],
    trace_globs=CONFIG['trace_globs'], strict_schema=CONFIG['strict_schema'],
    redact_text=CONFIG['redact_text'],
    include_reasoning_in_viewer=CONFIG['include_reasoning_in_viewer'],
)
metrics = compute_metrics(experiment, gold_dir=CONFIG['gold_dir'],
                          repo_roots=CONFIG['repo_roots'], compare_dir=CONFIG['compare_dir'])
findings = classify_failures(experiment, metrics)
metrics.summary

{'total_runs': 76,
 'known_outcomes': 76,
 'resolved': 20,
 'unresolved': 56,
 'unknown_outcomes': 0,
 'resolved_rate': 0.2631578947368421,
 'outcome_coverage': 1.0,
 'wilson_95': [0.17729398962172482, 0.3718124454091297],
 'discovered': 76,
 'accepted': 76,
 'rejected': 0,
 'method_version': '1.0.0',
 'experiment_id': 'itr-1',
 'percentage_policy': 'Known outcomes only; counts and coverage accompany rates'}

## Single-attempt explorer

Select a task to inspect its timeline, tool calls and results, budget, agent-side tests, independent final verification, token usage, final patch, findings, and source evidence. Batch comparisons appear in the separate section below.

In [3]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
report = render_report(experiment, metrics, findings)

## Export reproducible artifacts

This cell writes both HTML reports and batch summary tables, alongside the normalized CSV and JSONL tables. The HTML files bundle Plotly and work offline.

In [4]:
output = export_report(experiment, metrics, findings, CONFIG['output_dir'])
print(f'Exports: {output}')
print(f'Single-attempt report: {output / "report.html"}')
print(f'Whole-batch report: {output / "batch_report.html"}')

Exports: /Users/akshay/kaggle/gemma4_eval/reports/itr-1
Single-attempt report: /Users/akshay/kaggle/gemma4_eval/reports/itr-1/report.html
Whole-batch report: /Users/akshay/kaggle/gemma4_eval/reports/itr-1/batch_report.html


## Metric availability and optional modules

Gold-function recall needs gold patches and reliable baseline symbol maps. Paired outcomes need a second batch. Missing prerequisites never become zero-valued scores.

In [5]:
display(metrics.metrics.groupby(['metric_id', 'availability']).size().rename('attempts').reset_index())
display(metrics.modules)

,metric_id,availability,attempts
0,C01,available,76
1,C02,available,76
2,C03,partial,76
3,C04,unavailable,76
4,C05,unavailable,76
...,...,...,...
77,tool_attempts,available,76
78,tool_errors,available,76
79,tool_result_records,available,76
80,turn_count,available,76


,module,availability,prerequisites
0,gold_localization,unavailable,Gold patch + pristine Python baseline + return...
1,patch_scope,available,Final unified diff; file counts and byte size ...
2,graph_retrieval,available,Graph tool events; semantic retrieval quality ...
3,edit_survival,unavailable,Intermediate file snapshots and final-line pro...
4,test_recovery,partial,Ordered agent-side test output + edit evidence...
5,reasoning_judge,unavailable,Optional calibrated manual/judge rubric; no re...
6,reliability,partial,Wilson resolution interval; repeated attempts ...
7,cross_config,unavailable,Second batch + researcher confirmation of comp...


## Whole-batch report

The same notebook now produces a second report focused on aggregate outcomes, source-edit and verification coverage, budget exhaustion, repeated submissions, tool reliability, time/tokens, and data quality. All charts come from the normalized tables above. The separate HTML file is `../reports/itr-1/batch_report.html`. The task index links back to the detailed trace explorer.

In [6]:
from agent_eval import summarize_batch
from agent_eval.plots import render_batch_report

batch = summarize_batch(experiment, metrics, findings)
batch_view = render_batch_report(batch)
print(f'Whole-batch HTML: {output / "batch_report.html"}')

Whole-batch HTML: /Users/akshay/kaggle/gemma4_eval/reports/itr-1/batch_report.html
